# Reddit Script

Based on [PushshiftDumps/scripts/combine_folder_multiprocess.py](https://github.com/Watchful1/PushshiftDumps/blob/master/scripts/combine_folder_multiprocess.py)

In [ ]:
import urllib.request
import zstandard as zstd
import json
import os
import pandas as pd
from query_matcher import build_strata_matchers, classify
from query import ANTI, ISRAEL, PALESTINE

CAPS = {"anti_only": 50000, "ip_not": 30000, "ip_and": 20000}
DATA_DIR = "data/reddit"

def stream_month(url, file_type, month_label, output_dir=DATA_DIR):
    strata_matchers = build_strata_matchers(ANTI, ISRAEL, PALESTINE)
    counts = {stratum: 0 for stratum in strata_matchers}
    buffers = {stratum: [] for stratum in strata_matchers}
    os.makedirs(output_dir, exist_ok=True)

    cctx = zstd.ZstdDecompressor(max_window_size=2**31)
    lines_seen = 0
    lines_matched = 0

    req = urllib.request.Request(url, headers={"User-Agent": "research-collection/1.0"})
    with urllib.request.urlopen(req) as response:
        with cctx.stream_reader(response) as reader:
            buffer = ""
            while True:
                if all(counts[s] >= CAPS[s] for s in CAPS):
                    print(f"All strata capped for {month_label} ({file_type}), stopping early.")
                    break

                chunk = reader.read(2**24)
                if not chunk:
                    break
                buffer += chunk.decode("utf-8", errors="ignore")
                lines = buffer.split("\n")
                buffer = lines[-1]

                for line in lines[:-1]:
                    lines_seen += 1
                    if not line.strip():
                        continue
                    try:
                        post = json.loads(line)
                    except json.JSONDecodeError:
                        continue

                    if file_type == "comments":
                        text = post.get("body", "") or ""
                    else:
                        text = (post.get("title", "") or "") + " " + (post.get("selftext", "") or "")

                    if not text.strip():
                        continue

                    stratum = classify(text, strata_matchers)
                    if stratum is not None and counts[stratum] < CAPS[stratum]:
                        buffers[stratum].append({
                            "_source.text": text,
                            "id": post.get("id"),
                            "author": post.get("author"),
                            "subreddit": post.get("subreddit"),
                            "created_utc": post.get("created_utc"),
                            "score": post.get("score"),
                            "platform": "reddit",
                            "content_type": file_type,
                            "raw": line,
                        })
                        counts[stratum] += 1
                        lines_matched += 1

                    if lines_seen % 500000 == 0:
                        print(f"{month_label} {file_type}: {lines_seen:,} scanned, "
                              f"{lines_matched:,} matched, counts={counts}")

    for stratum, rows in buffers.items():
        if not rows:
            continue
        df = pd.DataFrame(rows)
        out_path_parquet = os.path.join(output_dir, f"{stratum}_{file_type}_{month_label}.parquet")
        out_path_csv = os.path.join(output_dir, f"{stratum}_{file_type}_{month_label}.csv")
        df.to_parquet(out_path_parquet, engine="pyarrow", index=False)
        df.to_csv(out_path_csv, index=False)
        print(f"Saved {len(df)} rows to {out_path_parquet} and {out_path_csv}")

    print(f"Done: {month_label} {file_type} | scanned {lines_seen:,} | matched {lines_matched:,} | final counts={counts}")
    return counts

In [ ]:
JOBS = [
    # (url, file_type, month_label)
    ("", "comments", "2025-01"),
]

all_results = {}
for url, file_type, month_label in JOBS:
    print(f"\n=== Starting {month_label} {file_type} ===")
    counts = stream_month(url, file_type, month_label)
    all_results[(month_label, file_type)] = counts

print("\n=== Summary ===")
for (month_label, file_type), counts in all_results.items():
    print(f"{month_label} {file_type}: {counts}")